# A tour of warpdrive

warpdrive is a thin layer over CuTeDSL for warp-specialized Blackwell kernels. You write each
role (load, MMA, epilogue, ...) as a method and connect roles with declared channels; the
layer derives barrier counts and phases, TMA byte arming, role dispatch, `setmaxnreg` and
warpgroup padding, TMEM allocation and the cross-CTA plumbing, and checks the protocol while
tracing. Tiles, layouts, stage counts, warps and the math stay yours, in ordinary CuTeDSL.

The tour builds up from a two-warp producer/consumer kernel to a pipelined TMA → UMMA →
epilogue GEMM, then covers scheduling, clusters, storage aliasing, the checkers and
`describe()`, and ends by mapping `persistent_gemm.py` and `fa4_fwd.py` onto the nouns.

| § | Nouns | Verbs |
|---|---|---|
| 1 | `WarpSpecialized`, `@role`, `RoleConfig`, `Ring`, `Threads`/`Warps`, `Producer`/`Consumer`, `Handle`, `Token`, `Preview`, `Slots`, `Sharing`, `NamedBarrier`, `Turnstile` | `acquire`/`commit`/`produce`, `wait`/`release`/`consume`, `try_wait`, `peek`, `describe()` |
| 2 | `Tma`, `Umma`, `SmemPlan`, `StageView`, `TmemPlan`, `Tmem` | `h.view`, `h.barrier`, `tmem.ptr` |
| 3 | `WorkStream`, `WorkMap`, `Raster`, schedules, `WorkCursor` | `work.item`, `work.advance()`, `bind`, `launch_args` |
| 4 | `CtaPair`, `Cluster`, `cta="leader"`, multicast `Tma` | `cluster.multicast` |
| 5 | `overlaps=`, `TmemPlan.alias`, access annotations | `alias_check` |
| 6 | `ProtocolError`, protocol trace | `trace_protocol`, `unchecked` |
| 7 | `raw=True` roles, `external=`, `override=`, `Barriers` | `signal=False`, `cute.arch` |
| 8 | `persistent_gemm.py`, `fa4_fwd.py` | |

Run it on a Blackwell GPU (written against a GB300, `sm_103a`, with `nvidia-cutlass-dsl`
4.8). On a shared host, reserve the GPU and bound the run:

```bash
gpu-run --timeout 300 auto -- timeout --signal=TERM --kill-after=5s 900s \
    jupyter nbconvert --execute --to notebook --inplace \
    transformer_nuggets/cute/warpdrive/docs/tour.ipynb
```

Kernels defined in this notebook compile with `compile_tvm_ffi` directly, never through
`jit_cache`: the cache keys a kernel class by the source file of its module, and a notebook
cell has none (see the note in §1). The library's examples use their cached helpers.

In [1]:
import inspect
import os
import re
import tempfile
import warnings
from dataclasses import dataclass
from types import SimpleNamespace

# CuTeDSL writes KeepPTX artifacts into the current directory unless told otherwise.
os.environ.setdefault("CUTE_DSL_DUMP_DIR", tempfile.mkdtemp(prefix="warpdrive_tour_"))

import cutlass
import torch
import torch.nn.functional as F
from cutlass import Int32, Int64, cute

from transformer_nuggets.cute.utils import compile_tvm_ffi, make_fake_compact_tensor
from transformer_nuggets.cute.warpdrive import (
    AtomicPersistent,
    Cluster,
    ClusterLaunchControl,
    Consumer,
    CtaPair,
    Instance,
    NamedBarrier,
    OneShot,
    Producer,
    ProtocolError,
    Raster,
    Ring,
    RoleConfig,
    Sharing,
    Slots,
    StaticPersistent,
    Threads,
    Tma,
    Turn,
    Turnstile,
    Umma,
    Warps,
    WarpSpecialized,
    WorkCursor,
    WorkMap,
    WorkStream,
    role,
)
from transformer_nuggets.utils.benchmark import (
    benchmark_cuda_function_in_microseconds as bench_us,
)

print(torch.cuda.get_device_name(), torch.cuda.get_device_capability(), cutlass.__version__)
torch.manual_seed(0)


def show(op: WarpSpecialized, *sections: str) -> None:
    """Print the ``describe()`` blocks whose title starts with one of ``sections``."""
    blocks = op.describe().split("\n\n")
    print("\n\n".join(b for b in blocks if b.startswith(sections)))


def show_channels(op: WarpSpecialized, *names: str) -> None:
    """Print only the named channels of ``describe()``'s Channels block."""
    text = op.describe()
    block = text[text.index("\nChannels\n") + 1 :].split("\n\n")[0].splitlines()[1:]
    keep = False
    for line in block:
        if not line.startswith("   "):  # a channel's first line; its details are indented
            keep = line.split()[0] in names
        if keep:
            print(line)


SYNC = re.compile(
    r"mbarrier\.(init|arrive|try_wait)|fence\.(mbarrier|proxy)|bar\.(warp\.)?sync|"
    r"barrier\.(sync|arrive)|setmaxnreg|tcgen05\.(alloc|dealloc|relinquish|commit)|"
    r"cp\.async\.bulk\.tensor|elect"
)


def compile_with_ptx(op, *fake_args, name: str):
    """``compile_tvm_ffi`` plus CuTeDSL's ``KeepPTX``, for reading what the layer emitted."""
    op.__call__.set_name_prefix(name)
    stream = cute.runtime.make_fake_stream(use_tvm_ffi_env_stream=True)
    return cute.compile[cute.EnableTVMFFI, cute.KeepPTX](op, *fake_args, stream)


def sync_ops(ptx: str) -> None:
    """Print the first instance of each synchronization instruction shape, in PTX order."""
    seen = {}
    for line in ptx.splitlines():
        op = line.strip()
        if SYNC.search(op):
            seen.setdefault(re.sub(r"%\w+|\d+", "#", op.split()[0]), op)
    print("\n".join(seen.values()))


def tflops(flops: float, us: float) -> str:
    return f"{us:8.1f} us  {flops / us / 1e6:6.0f} TFLOP/s"

NVIDIA GB300 (10, 3) 4.8.0


## 1. The smallest kernel: two roles and a ring

A `WarpSpecialized` subclass declares channels as attributes in `__init__` and roles as
methods decorated `@role(warps=...)` directly above `@cute.jit`. `warps=0` means warp
index 0, not zero warps. A role parameter annotated `Producer` or `Consumer` receives that
end of the channel with the same name; the single unannotated parameter receives the
context passed to `run_roles`. Every role returns the ends it took.

`Ring(2, Threads, Threads, payload=...)` reads: two reusable stages; every producer thread
arrives to publish a stage; every consumer thread arrives to hand it back. The payload is
layer-owned SMEM of shape `(32, stages)`; `h.data` is the handle's stage. The producer below
spells the verbs out (`acquire` → write → `commit`); the consumer uses the context-manager
form (`consume()` = `wait` on entry, `release` on exit). `style` swaps in the other
consumer verbs used later in this section.

In [2]:
WORDS = 32  # one Int32 per lane per stage


class Relay(WarpSpecialized):
    """Warp 0 writes epoch e's words ``e * 32 + lane``; the consumer sums them per lane."""

    def __init__(self, stages: int = 2, release=Threads, style: str = "with", roles=None):
        self.style = style
        self.roles = roles  # {role name: RoleConfig} over the @role defaults
        self.ch = Ring(stages, Threads, release, payload=(Int32, (WORDS,)))

    @role(warps=0)
    @cute.jit
    def producer(self, ctx, *, ch: Producer):
        lane = cute.arch.lane_idx()
        for e in cutlass.range(ctx.n, unroll=1):
            h = ch.acquire()  # wait until stage h.index is empty; the end moves on
            h.data[lane] = e * WORDS + lane
            h.commit()  # publish: every producer thread arrives on FULL
        return ch

    @role(warps=1)
    @cute.jit
    def consumer(self, ctx, *, ch: Consumer):
        lane = cute.arch.lane_idx()
        total = Int32(0)
        for _ in cutlass.range(ctx.n, unroll=1):
            if cutlass.const_expr(self.style == "token"):
                token = ch.try_wait()  # one probe of FULL; no handle, no advance
                h = ch.wait(token=token)  # blocks only if the probe saw the stage empty
                total += h.data[lane]
                h.release()
            elif cutlass.const_expr(self.style == "peek"):
                preview = ch.peek()  # waits for FULL, read-only, does not advance
                total += preview.view()[lane]
                h = ch.wait()  # owns the previewed epoch; its release covers the read
                h.release()
            else:
                with ch.consume() as h:
                    total += h.data[lane]
        ctx.out[lane] = total  # every consumer warp writes the same sums
        return ch

    @cute.kernel
    def kernel(self, out: cute.Tensor, n: Int32):
        self.begin_roles()  # allocate + initialize barriers and payloads
        self.run_roles(SimpleNamespace(out=out, n=n))  # fence, sync, dispatch, teardown

    @cute.jit
    def __call__(self, out: cute.Tensor, n: Int32, stream):
        self.kernel(out, n).launch(**self.launch_config(grid=(1, 1, 1)), stream=stream)


print(Relay().describe())

Relay: 2 warps, 64 threads
  cluster  single CTA, tcgen05 CtaGroup.ONE
  params   unannotated (any object)

Roles
  warps  role      regs  cta  raw
  0      producer     -  all
  1      consumer     -  all

Channels
  ch  Ring(2)  producer -> consumer
      signal    Threads -> Threads
      full      32 x2
      empty     32 x2
      storage   payload Int32(32,)


`describe()` runs before anything is compiled: it collects the declarations and prints
what the layer derived. `full 32 x2` means each of the two FULL barriers expects 32
arrivals (the producer warp's threads); `empty 32 x2` is the consumer's 32.

Compile with fake compact tensors (dynamic sizes stay runtime values), launch, and check
against a host reference. `n` is a runtime epoch count; 1, 2 and 7 exercise the first
stage, both stages, and phase wraparound.

Note on caching: the library's examples compile through `jit_cache`, which keeps kernels on
disk keyed by the constructor call and the source of the kernel's module. A class defined
in a notebook cell has no module file, so its source would not reach the key and an edited
cell could load a stale build. The notebook's own kernels therefore call `compile_tvm_ffi`
directly.

In [3]:
def build_relay(op: Relay):
    out = make_fake_compact_tensor(Int32, (WORDS,), assumed_align=4)
    return compile_tvm_ffi(op, out, Int32(1), name="tour_relay")


def check_relay(fn, n: int) -> None:
    out = torch.full((WORDS,), -1, device="cuda", dtype=torch.int32)
    fn(out, n)
    lane = torch.arange(WORDS, device="cuda", dtype=torch.int32)
    torch.testing.assert_close(out, WORDS * n * (n - 1) // 2 + n * lane)


relay = build_relay(Relay())
for n in (1, 2, 7, 1000):
    check_relay(relay, n)
print("Relay matches the reference for n = 1, 2, 7, 1000")

Relay matches the reference for n = 1, 2, 7, 1000


### What the layer generated

The managed verbs lower to a handful of mbarrier instructions. `begin_roles` emits one
`mbarrier.init` per barrier (count 32, from one lane of warp 0); `run_roles` adds the init
fence and a CTA barrier before dispatch; `acquire`/`wait` emit `try_wait.parity` on the
EMPTY/FULL barrier of the end's current stage and phase (its `PipelineState`);
`commit`/`release` emit one `mbarrier.arrive` per thread.

In [4]:
relay_ptx = compile_with_ptx(
    Relay(), make_fake_compact_tensor(Int32, (WORDS,), assumed_align=4), Int32(1), name="relay"
).__ptx__
sync_ops(relay_ptx)

mbarrier.init.shared.b64 	[%r17], %r18;
fence.mbarrier_init.release.cluster;
bar.sync 	0;
mbarrier.try_wait.parity.shared.b64 P1, [%r20], %r39, 10000000;
mbarrier.arrive.shared.b64 %rd2, [%r24], %r27;


By hand, the same kernel is the classic raw protocol. A sketch (not run) of what
`begin_roles`, `run_roles` and the two ends replace, using the CuTeDSL calls
`examples/raw_interop.py` uses for its raw consumer:

```python
bars = smem.allocate_array(cutlass.Int64, 2 * STAGES)  # FULL at [0, n), EMPTY at [n, 2n)
if warp == 0:
    with cute.arch.elect_one():
        for s in range(STAGES):
            cute.arch.mbarrier_init(bars + s, 32)  # FULL: 32 producer threads
            cute.arch.mbarrier_init(bars + STAGES + s, 32)  # EMPTY: 32 consumer threads
cute.arch.mbarrier_init_fence()
cute.arch.sync_threads()
if warp == 0:  # producer: EMPTY starts "free", so its phase starts at 1
    st = pipeline.make_pipeline_state(pipeline.PipelineUserType.Producer, STAGES)
    for e in cutlass.range(n, unroll=1):
        cute.arch.mbarrier_wait(bars + STAGES + st.index, st.phase)
        buf[lane, st.index] = e * 32 + lane
        cute.arch.mbarrier_arrive(bars + st.index)
        st.advance()
elif warp == 1:  # consumer: mirror image, phase starts at 0
    ...
```

The counts (32, 32), the barrier offsets and the initial phases are the parts warpdrive
derives. They change when the roles change; the next cell changes them.

### Mechanisms and placement change the counts, not the role bodies

`Threads` counts every thread of the bound roles; `Warps` has every lane run the side's
fence and `bar.warp.sync`, then one elected lane per warp arrives. `RoleConfig` replaces a
role's `@role` placement from the constructor (`self.roles`); moving the consumer to warps
4–7 makes it a four-warp role. A `regs=` budget is per warpgroup (`setmaxnreg`): roles that
set one pad the launch to whole warpgroups, and the padding warps take their group's budget.

The consumer's other verbs: `try_wait()` probes FULL once and returns a `Token`;
`wait(token=...)` skips the blocking wait when the probe succeeded. `peek()` waits for the
next epoch and returns a read-only `Preview` without advancing; the next `wait()` owns that
epoch. Both are checked while tracing: a stale token or a preview with no owning wait is a
`ProtocolError`.

In [5]:
wide = Relay(release=Warps, roles={"consumer": RoleConfig(range(4, 8))})
show(wide, "Relay", "Roles", "Channels")
budgets = Relay(
    roles={"producer": RoleConfig(0, regs=40), "consumer": RoleConfig(range(4, 8), regs=232)}
)
print()
show(budgets, "Roles")
print()

for name, op in [
    ("four-warp Warps consumer", wide),
    ("try_wait token", Relay(style="token")),
    ("peek preview", Relay(style="peek")),
]:
    fn = build_relay(op)
    for n in (1, 7, 1000):
        check_relay(fn, n)
    print(f"{name}: matches")

print("\nWarps release, elected per warp after a warp sync:")
fake_out = make_fake_compact_tensor(Int32, (WORDS,), assumed_align=4)
wide_ptx = compile_with_ptx(
    Relay(release=Warps, roles={"consumer": RoleConfig(range(4, 8))}),
    fake_out,
    Int32(1),
    name="relay_warps",
).__ptx__
sync_ops(wide_ptx)

Relay: 8 warps, 256 threads
  cluster  single CTA, tcgen05 CtaGroup.ONE
  params   unannotated (any object)

Roles
  warps  role      regs  cta  raw
  0      producer     -  all
  4-7    consumer     -  all

Channels
  ch  Ring(2)  producer -> consumer
      signal    Threads -> Warps
      full      32 x2
      empty     4 x2
      storage   payload Int32(32,)

Roles
  warps  role       regs  cta  raw
  0      producer     40  all
  1-3    (padding)    40
  4-7    consumer    232  all



four-warp Warps consumer: matches


try_wait token: matches


peek preview: matches

Warps release, elected per warp after a warp sync:


mbarrier.init.shared.b64 	[%r17], %r18;
fence.mbarrier_init.release.cluster;
bar.sync 	0;
mbarrier.try_wait.parity.shared.b64 P1, [%r21], %r41, 10000000;
mbarrier.arrive.shared.b64 %rd2, [%r25], %r28;
bar.warp.sync 	-1;
elect.sync 	%r36|%p9, -1;


With `Warps` the EMPTY count drops from 128 (four warps × 32 threads) to 4, and the release
gains a `bar.warp.sync` and an `elect.sync` before its arrive; the FULL side is unchanged. With budgets, warps 1–3 run no role but share warp 0's warpgroup, so they take
its 40 registers. `describe()` rejects a split that does not fit the registers the launch
starts with (`setmaxnreg` only moves registers).

### The rest of the channel vocabulary

A declaration-only kernel (role bodies just return their state) is enough to see what the
layer derives for the other channel shapes. `describe()` needs no compile.

- `Slots(n, ...)`: the role picks the slot (`s.wait(j)`); `Consumer[Instance]` binds each
  instance of a multi-instance role to its own slot. `softmax[0]@0` in the output.
- `Sharing.ROUND_ROBIN`: instances take a side's epochs in turn (one instance's count);
  `Sharing.EXCLUSIVE` gives each epoch to one of several consumer roles, which `skip()` the rest.
- `sync="named"`: a Threads → Threads channel on CTA named barriers instead of mbarriers.
- `NamedBarrier(threads=N)`: one allocated barrier ID for a rendezvous or a one-shot flag.
- `Turnstile`: a round-robin critical section across a role's instances (`with mufu.turn():`).

In [6]:
class Vocabulary(WarpSpecialized):
    def __init__(self):
        self.s = Slots(2, Umma, Threads)  # MMA fills slot j, softmax[j] reads it
        self.rr = Ring(4, Threads, Threads, consumer_sharing=Sharing.ROUND_ROBIN)
        self.row_max = Ring(1, Threads, Threads, sync="named")
        self.done = NamedBarrier(threads=256 + 32)
        self.mufu = Turnstile()

    @role(warps=8)
    @cute.jit
    def mma(self, ctx, *, s: Producer, rr: Producer, row_max: Producer):
        return s, rr, row_max

    @role(warps=[range(4), range(4, 8)])
    @cute.jit
    def softmax(
        self,
        ctx,
        *,
        stage: Instance,
        s: Consumer[Instance],
        rr: Consumer,
        row_max: Consumer,
        mufu: Turn,
    ):
        return s, rr, row_max, mufu


show(Vocabulary(), "Roles", "Channels", "Named", "Turnstiles")

Roles
  warps  role        regs  cta  raw
  0-3    softmax[0]     -  all
  4-7    softmax[1]     -  all
  8      mma            -  all

Channels
  s        Slots(2) mma -> softmax[0]@0, softmax[1]@1
           signal    Umma -> Threads
           full      1 x2
           empty     128 x2
  rr       Ring(4)  mma -> softmax[0], softmax[1]
           signal    Threads -> Threads
           full      32 x4
           empty     128 x4
           turns     consumers round-robin: instance j of 2 takes epochs j, j+2, ...; counts are one instance's
           epochs    ownership not traced
  row_max  Ring(1)  mma -> softmax[0], softmax[1]
           signal    Threads -> Threads
           named     full ids [14], empty ids [15], 288 threads

Turnstiles
  mufu  softmax[0] -> softmax[1]

Named barriers
  done  id 3  288 threads  users known after a compile


## 2. A pipelined GEMM: TMA → UMMA → epilogue

[`examples/gemm.py`](../examples/gemm.py) computes BF16 `D = A @ B.T`, one 128 × 256 tile per
CTA: warp 4 loads A/B with TMA, warp 5 issues `tcgen05` MMAs, warps 0–3 read the accumulator
from TMEM and store D. Two channels carry the handoffs:

| Channel | Declaration | Producer → consumer | Epochs per tile |
|---|---|---|---|
| `ab` | `Ring(3, Tma(49152), Umma, buffer={"A": a, "B": b})` | TMA bytes complete FULL → MMA completion (`tcgen05.commit`) frees the stage | `ceil(K / 64)` |
| `acc` | `Ring(1, Umma, Threads)` | MMA completion publishes → 128 epilogue threads release | 1 |

New nouns:

- `Tma(bytes)`: acquire arms the FULL barrier with the stage's byte count; the copies
  (passed `tma_bar_ptr=h.barrier`) complete it. `Umma`: an elected `tcgen05.commit` signals
  when the issuing thread's MMAs finish, so the stage is reused only after the MMA read it.
- `SmemPlan` declares named buffers; `buffer=` binds them to a channel and `begin_roles`
  allocates them from the layouts you pass. `StageView(tensor, axis)` registers a TMA or MMA
  partition's stage mode; `h.view("A")` selects the handle's stage of it.
- `TmemPlan` names TMEM column ranges; a role taking `tmem: Tmem` gets `tmem.ptr("acc")` and
  joins the coordinated free. The MMA holds `acc.produce()` around the whole K loop.

The MMA role is the whole nesting story:

In [7]:
from transformer_nuggets.cute.warpdrive.examples.gemm import Gemm, gemm

print(inspect.getsource(Gemm.mma))
print(Gemm().describe())

    @role(warps=5, regs=40)
    @cute.jit
    def mma(
        self, params: GemmParams, *, ab: Consumer, acc: Producer, work: WorkCursor, tmem: Tmem
    ):
        tiled_mma = params.tiled_mma
        tCtAcc = cute.make_tensor(tmem.ptr("acc"), params.acc_layout)
        ab = ab.views(A=params.tCrA, B=params.tCrB)
        while work.valid:
            with acc.produce() as a:
                tiled_mma.set(tcgen05.Field.ACCUMULATE, False)
                for k in cutlass.range(params.num_k_tiles, unroll=1):
                    with ab.consume() as s:
                        for kb in cutlass.range_constexpr(cute.size(params.tCrA.tensor, mode=[2])):
                            cute.gemm(
                                tiled_mma,
                                tCtAcc[None, None, None, a.index],
                                s.view("A", None, None, kb),
                                s.view("B", None, None, kb),
                                tCtAcc[None, None, None, a.index],
      

`tx 49152 B` is `(128 + 256) × 64 × 2` bytes per stage. `tma not checked yet` and the
missing SMEM sizes are what tracing fills in: the layouts bind in `begin_roles`, and the
TMA check runs on the traced load role (next cells). The 160/40/40 budgets pad the launch
to two warpgroups, so warps 6–7 run nothing.

Correctness against an FP32 reference, with PyTorch's BF16 error as the yardstick, then a
timing at 8192³ (profiler kernel time, median; warm L2, fixed tensors, default clocks):

In [8]:
def gemm_error(fn, m: int, n: int, k: int, **kwargs) -> tuple[float, float]:
    a = torch.randn(m, k, device="cuda", dtype=torch.bfloat16)
    b = torch.randn(n, k, device="cuda", dtype=torch.bfloat16)
    ref = a.float() @ b.float().T
    err = (fn(a, b, **kwargs).float() - ref).abs().max().item()
    return err, ((a @ b.T).float() - ref).abs().max().item()


for shape in [(512, 512, 512), (300, 520, 136), (4097, 2056, 4104)]:
    err, base = gemm_error(gemm, *shape)
    print(f"M, N, K = {shape}: max |err| {err:.3g} (torch bf16 {base:.3g})")
    assert err <= base * 1.01 + 1e-3

a8 = torch.randn(8192, 8192, device="cuda", dtype=torch.bfloat16)
b8 = torch.randn_like(a8)
flops8 = 2 * 8192**3
print("warpdrive gemm ", tflops(flops8, bench_us(gemm, a8, b8)))
print("torch.mm       ", tflops(flops8, bench_us(torch.mm, a8, b8.T)))

M, N, K = (512, 512, 512): max |err| 0.25 (torch bf16 0.25)
M, N, K = (300, 520, 136): max |err| 0.125 (torch bf16 0.125)
M, N, K = (4097, 2056, 4104): max |err| 1 (torch bf16 1)
warpdrive gemm     857.0 us    1283 TFLOP/s


torch.mm           617.9 us    1780 TFLOP/s


This GEMM is the teaching kernel: one tile per CTA, no epilogue/mainloop overlap. The
persistent GEMM in §4 closes most of the gap to cuBLAS.

### What the layer generated for the GEMM

The PTX shows each derived piece: `mbarrier.init` per barrier, the TMEM allocation (by a
TMEM-using warp other than warp 0, which initializes the barriers) and its permit release,
`setmaxnreg` per warpgroup, the teardown barrier (ID 2, 160 threads: the TMEM users) before
`dealloc`, `expect_tx` arming 49152 bytes before the TMA copies, and the UMMA release as
`tcgen05.commit ... mbarrier::arrive::one`. After this traced compile, `describe()` reports
the TMA check.

In [9]:
def gemm_fakes():
    m, n, k = cute.sym_int(), cute.sym_int(divisibility=8), cute.sym_int(divisibility=8)
    return tuple(
        make_fake_compact_tensor(cutlass.BFloat16, shape) for shape in ((m, k), (n, k), (m, n))
    )


gemm_op = Gemm()
sync_ops(compile_with_ptx(gemm_op, *gemm_fakes(), name="gemm").__ptx__)
print()
show_channels(gemm_op, "ab")

mbarrier.init.shared.b64 	[%r190], %r191;
tcgen05.alloc.cta_group::1.sync.aligned.shared::cta.b32 	[%r8], %r193;
tcgen05.relinquish_alloc_permit.cta_group::1.sync.aligned;
fence.mbarrier_init.release.cluster;
bar.sync 	0;
setmaxnreg.dec.sync.aligned.u32 	160;
mbarrier.try_wait.parity.shared.b64 P1, [%r9], %r213, 10000000;
mbarrier.arrive.shared.b64 %rd146, [%r15], %r988;
barrier.sync 	2, 160;
tcgen05.dealloc.cta_group::1.sync.aligned.b32 	%r989, %r990;
elect.sync 	%r1043|%p819, -1;
mbarrier.arrive.expect_tx.shared.b64 %rd155, [%r146], %r1044;
cp.async.bulk.tensor.2d.shared::cta.global.tile.mbarrier::complete_tx::bytes.L2::cache_hint [%r148+1024], [%rd156, {%r1051, %r144}], [%r146], %rd157;
tcgen05.commit.cta_group::1.mbarrier::arrive::one.shared::cluster.b64 	[%r159];

  ab   Ring(3)  load -> mma
       signal    Tma(49152 B) -> Umma
       full      1 x3, tx 49152 B
       empty     1 x3
       storage   buffer A, B
       tma       verified CtaGroup.ONE, 49152 B/stage per CTA @ gemm.

The CUTLASS-style setup these declarations replace (sketch, not run):

```python
ab = pipeline.PipelineTmaUmma.create(
    num_stages=3,
    producer_group=pipeline.CooperativeGroup(pipeline.Agent.Thread),
    consumer_group=pipeline.CooperativeGroup(pipeline.Agent.Thread, 1),
    tx_count=49152,  # must equal the bytes the copies deliver
    barrier_storage=storage.ab_bars.data_ptr(),
)
acc = pipeline.PipelineUmmaAsync.create(
    num_stages=1,
    producer_group=pipeline.CooperativeGroup(pipeline.Agent.Thread),
    consumer_group=pipeline.CooperativeGroup(pipeline.Agent.Thread, 128),  # epilogue threads
    barrier_storage=storage.acc_bars.data_ptr(),
)
# plus: warp-index dispatch, setmaxregister_{increase,decrease} per warpgroup, TMEM
# alloc/relinquish/dealloc around a named barrier, producer_tail, and one PipelineState per
# role kept in step by hand
```

Here `tx_count=49152` and the `128` are facts about the copies and the epilogue's warps
that the code must restate. warpdrive computes them from the declared roles, and the TMA
check compares the copies against the declared bytes (§6 shows the error when they differ).

## 3. Scheduling: which work each CTA runs

`self.work = WorkStream(mapping=..., schedule=...)` gives every role of a work unit (a CTA,
or a whole cluster) the same sequence of items. The **schedule** picks linear indices for
each unit; the **mapping** (`WorkMap`) decodes an index into an item. A role takes
`work: WorkCursor` and drains it with exactly this loop:

```python
while work.valid:
    m, n, _ = work.item  # read before advancing
    ...
    work.advance()  # once, at the end of the body
return ..., work
```

The toy below records which CTA ran each item and in what order. `LowerTriangle` is a
custom `WorkMap`: it decodes a raster index and drops tiles above the diagonal
(`skips = True`), as a causal mask would.

In [10]:
@dataclass(frozen=True)
class LowerTriangle(WorkMap):
    """Raster over ``(m_tiles, n_tiles, 1)``, m fastest, keeping tiles with ``n <= m``."""

    words = 3
    skips = True  # decode may drop an index for every role

    def count64(self, space):
        m_tiles, n_tiles, _ = space
        return Int64(m_tiles) * n_tiles

    def decode(self, index, space, state):
        m_tiles = space[0]
        m, n = index % m_tiles, index // m_tiles
        return n <= m, (m, n, 0), state  # (keep, words, next state)

    def get_key(self) -> str:
        return "lower_triangle"


class WhoRunsWhat(WarpSpecialized):
    def __init__(self, schedule, mapping=None):
        self.work = WorkStream(mapping=mapping or Raster(), schedule=schedule)

    @role(warps=0)
    @cute.jit
    def worker(self, ctx, *, work: WorkCursor):
        bx, by, _ = cute.arch.block_idx()
        gx, _, _ = cute.arch.grid_dim()
        cta, step = bx + by * gx, Int32(0)  # linear CTA id (OneShot launches a 2-D grid)
        while work.valid:
            m, n, _ = work.item
            if cute.arch.lane_idx() == 0:
                ctx.owner[m, n] = cta
                ctx.order[m, n] = step
            step += 1
            work.advance()
        return work

    @cute.kernel
    def kernel(self, owner: cute.Tensor, order: cute.Tensor, bound):
        self.begin_roles(work=bound)
        self.run_roles(SimpleNamespace(owner=owner, order=order))

    @cute.jit
    def __call__(
        self,
        owner: cute.Tensor,
        order: cute.Tensor,
        workspace: cute.Tensor,
        max_ctas: Int32,
        stream,
    ):
        bound = self.work.bind((owner.shape[0], owner.shape[1], 1), workspace, max_ctas)
        self.kernel(owner, order, bound).launch(**self.launch_config(bound), stream=stream)


def who_runs_what(schedule, mapping=None, shape=(4, 6), max_ctas=None):
    op = WhoRunsWhat(schedule, mapping)
    grid = make_fake_compact_tensor(Int32, (cute.sym_int(), cute.sym_int()), assumed_align=4)
    workspace = make_fake_compact_tensor(Int32, (cute.sym_int(),), assumed_align=4)
    fn = compile_tvm_ffi(op, grid, grid, workspace, Int32(1), name="tour_who_runs_what")
    owner = torch.full(shape, -1, device="cuda", dtype=torch.int32)
    order = torch.full_like(owner, -1)
    workspace, sms = op.work.launch_args(owner.device)  # (zeroed counter, resident CTAs)
    fn(owner, order, workspace, max_ctas or sms)
    return op, owner.cpu(), order.cpu()


for label, schedule, mapping in [
    ("OneShot(), Raster()", OneShot(), None),
    ("StaticPersistent(), Raster()", StaticPersistent(), None),
    ("StaticPersistent(), Raster(swizzle=2)", StaticPersistent(), Raster(swizzle=2)),
    ("StaticPersistent(), LowerTriangle()", StaticPersistent(), LowerTriangle()),
]:
    _, owner, order = who_runs_what(schedule, mapping, max_ctas=3)
    print(f"{label}: CTA of each (m, n), then its visit order")
    for owner_row, order_row in zip(owner.tolist(), order.tolist()):
        print("  ", " ".join(f"{c:2d}" for c in owner_row), "   ", order_row)

OneShot(), Raster(): CTA of each (m, n), then its visit order
    0  4  8 12 16 20     [0, 0, 0, 0, 0, 0]
    1  5  9 13 17 21     [0, 0, 0, 0, 0, 0]
    2  6 10 14 18 22     [0, 0, 0, 0, 0, 0]
    3  7 11 15 19 23     [0, 0, 0, 0, 0, 0]


StaticPersistent(), Raster(): CTA of each (m, n), then its visit order
    0  1  2  0  1  2     [0, 1, 2, 4, 5, 6]
    1  2  0  1  2  0     [0, 1, 3, 4, 5, 7]
    2  0  1  2  0  1     [0, 2, 3, 4, 6, 7]
    0  1  2  0  1  2     [1, 2, 3, 5, 6, 7]


StaticPersistent(), Raster(swizzle=2): CTA of each (m, n), then its visit order
    0  1  2  0  1  2     [0, 0, 2, 3, 5, 5]
    2  0  1  2  0  1     [0, 1, 3, 3, 6, 6]
    1  2  0  1  2  0     [1, 1, 4, 4, 6, 7]
    0  1  2  0  1  2     [2, 2, 4, 5, 7, 7]


StaticPersistent(), LowerTriangle(): CTA of each (m, n), then its visit order
    0 -1 -1 -1 -1 -1     [0, -1, -1, -1, -1, -1]
    1  2 -1 -1 -1 -1     [0, 1, -1, -1, -1, -1]
    2  0  1 -1 -1 -1     [0, 2, 2, -1, -1, -1]
    0  1  2  0 -1 -1     [1, 1, 2, 3, -1, -1]


`OneShot` launches the tile space, one CTA per item. `StaticPersistent` with three CTAs
strides the indices by 3; swizzling changes which item an index decodes to, not the
strides; `LowerTriangle` drops items from every role's sequence.

The broadcast schedules add a scheduler warp: `AtomicPersistent` claims from a global
counter (the `workspace` from `launch_args`), `ClusterLaunchControl` cancels CTAs that have
not launched yet and takes their work. The scheduler publishes each item to the roles over
a generated `work_ring` channel. On 65536 items, `AtomicPersistent` launches `max_ctas`
CTAs (the SM count `launch_args` estimates) that loop over the counter. CLC launches the whole
grid; each resident CTA keeps cancelling CTAs that have not launched and runs their items, so
about as many CTAs run as fit at once (this tiny kernel fits many per SM). The counts are a
runtime snapshot:

In [11]:
for schedule in (AtomicPersistent(), ClusterLaunchControl()):
    op, owner, _ = who_runs_what(schedule, shape=(256, 256))
    ran = torch.bincount(owner.flatten())
    print(
        f"{schedule.describe()}: {(ran > 0).sum().item()} CTAs ran the {owner.numel()} items, "
        f"at most {ran.max().item()} each"
    )
show(op, "Roles", "Channels", "Streams")

AtomicPersistent(stages=2): 152 CTAs ran the 65536 items, at most 536 each


ClusterLaunchControl(stages=2): 4864 CTAs ran the 65536 items, at most 31 each
Roles
  warps  role            regs  cta  raw
  0      worker             -  all
  1      work.scheduler     -  all

Channels
  work_ring  Ring(2)  work.scheduler -> worker, work.scheduler
             signal    Threads -> Threads
             full      32 x2
             empty     64 x2
             storage   payload Int32(4,)
  work_clc   Slots(1) work.scheduler -> work.scheduler
             signal    Tma(16 B) -> Threads
             full      1, tx 16 B
             empty     32
             storage   payload Int32(4,), release_fence smem_async

Streams
  work  ClusterLaunchControl(stages=2), Raster(along=m)
        scheduler warp 1 broadcasts over work_ring
        roles: worker


## 4. Clusters and CTA pairs

A cluster owns one work item. `self.cluster = CtaPair()` runs 2-CTA (`CtaGroup.TWO`) MMAs:
each CTA loads half the operands, the leader's MMA warp (`@role(..., cta="leader")`) issues
MMAs that read both CTAs' SMEM, and each epilogue stores its half from its own TMEM. The
declaration does not convert atoms or layouts (double the MMA tile's M, build atoms with
`self.cluster.cta_group`); it derives which barriers every signal reaches.

`Cluster(2, 2, pair=True)` adds a second pair along N. Both pairs need the same rows of A,
so each CTA loads part of the A tile and multicasts it: `Tma(bytes, multicast="n")`, with
masks from `self.cluster.multicast("n")`.

[`examples/persistent_gemm.py`](../examples/persistent_gemm.py) takes the schedule and the
cluster as constructor arguments and keeps one set of role bodies: `gemm.py`'s roles in a
work loop, two TMEM accumulator stages (the MMA starts tile i+1 while the epilogue drains
tile i) and a TMA-store epilogue. Compare the derived `ab` and `acc` channels:

In [12]:
from transformer_nuggets.cute.warpdrive.examples.persistent_gemm import (
    PersistentGemm,
    compile_persistent_gemm,
    persistent_gemm,
)

for cluster in (CtaPair(), Cluster(2, 2, pair=True)):
    op = PersistentGemm(cluster=cluster)
    compile_persistent_gemm(op)  # fills the TMA check line
    print(f"{cluster}:")
    show_channels(op, "ab", "acc")
    print()

CtaPair(m=2, n=1, pair=True):
  ab   Ring(6)  load -> mma
       signal    Tma(30720 B) -> Umma
       full      leader 1 x6, tx 61440 B (leader-armed)
       empty     leader 1 x6, peer 1 x6
       scope     pair-scoped, release_mask 0b11, tail
       storage   buffer A, B
       tma       verified CtaGroup.TWO, 30720 B/stage per CTA @ persistent_gemm.py:220,221
  acc  Ring(2)  mma -> epilogue
       signal    Umma -> Threads
       full      leader 1 x2, peer 1 x2
       empty     leader 256 x2
       scope     pair-scoped, commit_mask 0b11, remote_scope cta

Cluster(m=2, n=2, pair=True):
  ab   Ring(6)  load -> mma
       signal    Tma(32768 B) -> Umma
       full      rank 0 1 x6, rank 2 1 x6, tx 65536 B (leader-armed)
       empty     rank 0 2 x6, rank 1 2 x6, rank 2 2 x6, rank 3 2 x6
       scope     cross-CTA, multicast n, release_mask 0b1111, tail
       storage   buffer A, B
       tma       verified CtaGroup.TWO, 32768 B/stage per CTA @ persistent_gemm.py:220,221
  acc  Ring(

Reading the pair: the leader's FULL barrier collects both CTAs' bytes (`tx 61440 B
(leader-armed)`, twice the per-CTA 30720), the MMA's completion releases A/B in both CTAs
(`release_mask 0b11`) and publishes the accumulator to both (`commit_mask 0b11`), and the
leader's EMPTY barrier waits for both epilogues (`256` = 2 × 128 threads, which the role
code never states). `tail`: the producer drains its last stages before exit, because the
remote MMA may still be reading them.

In the 2 × 2 cluster each CTA's stage is overwritten by the other pair's next multicast, so
every CTA's EMPTY waits for both leaders (`2 x6`).

Every schedule and cluster below runs the same role bodies. Errors are on a ragged
4097 × 2056 × 4104 problem against FP32 (PyTorch BF16 for scale); timings as in §2, at
8192³, where `torch.mm` took the time printed there.

In [13]:
configs = [
    ("OneShot, 1 CTA", OneShot(), None),
    ("StaticPersistent, 1 CTA", StaticPersistent(), None),
    ("AtomicPersistent, 1 CTA", AtomicPersistent(), None),
    ("ClusterLaunchControl, 1 CTA", ClusterLaunchControl(), None),
    ("ClusterLaunchControl, CtaPair", ClusterLaunchControl(), CtaPair()),
    ("StaticPersistent, Cluster(2, 2)", StaticPersistent(), Cluster(2, 2, pair=True)),
]
for label, schedule, cluster in configs:
    kwargs = {"schedule": schedule, "cluster": cluster}
    err, base = gemm_error(persistent_gemm, 4097, 2056, 4104, **kwargs)
    assert err <= base * 1.01 + 1e-3, (label, err, base)
    us = bench_us(persistent_gemm, a8, b8, **kwargs)
    print(f"{label:32s} err {err:.3g} (torch {base:.3g})  {tflops(flops8, us)}")

OneShot, 1 CTA                   err 0.996 (torch 0.996)     694.0 us    1584 TFLOP/s


StaticPersistent, 1 CTA          err 0.997 (torch 0.997)     655.1 us    1678 TFLOP/s


AtomicPersistent, 1 CTA          err 0.998 (torch 0.998)     672.1 us    1636 TFLOP/s


ClusterLaunchControl, 1 CTA      err 0.998 (torch 0.998)     671.0 us    1639 TFLOP/s


ClusterLaunchControl, CtaPair    err 1 (torch 1)     628.3 us    1750 TFLOP/s


StaticPersistent, Cluster(2, 2)  err 0.997 (torch 0.997)     649.6 us    1693 TFLOP/s


## 5. Storage and aliasing

Channels order *when* a stage may be used; `SmemPlan` and `TmemPlan` say *where* it lives.
Overlapping storage is declared, never inferred:

- `smem_plan.buffer("V", ..., overlaps=k)` gives K's bytes a second layout (FA4's V reuses
  K's stages; both bind to one ring).
- `TmemPlan().region("S", ...).alias("P", of="S", ...)` places P inside S's columns.

Overlapping bytes does not order accesses. The alias diagnostic checks accesses you
annotate at the operation (`h.view("K", write=True, by="tma")`,
`tmem.access("P", j, read=True, by="umma")`) against the channel paths between them. It is a
bug finder: `no hazard found` is not a proof. FA4 forward declares both kinds of alias:

In [14]:
from transformer_nuggets.cute.warpdrive.examples.fa4_fwd import (
    Fa4Params,
    FlashAttentionForward,
    compile_fa4_fwd,
    fa4_fwd,
)

fa4 = FlashAttentionForward()
compile_fa4_fwd(fa4)
show(fa4, "TMEM", "SMEM", "Alias")

b_, s_, h_, d_ = 4, 4096, 16, 128
q, k, v = (torch.randn(b_, s_, h_, d_, device="cuda", dtype=torch.bfloat16) for _ in range(3))
out, lse = fa4_fwd(q, k, v)
ref = F.scaled_dot_product_attention(*(x.transpose(1, 2).float() for x in (q, k, v)))
sdpa = lambda: F.scaled_dot_product_attention(*(x.transpose(1, 2) for x in (q, k, v)))
err = (out.float() - ref.transpose(1, 2)).abs().max().item()
base = (sdpa().float() - ref).abs().max().item()
print(f"\nfa4_fwd max |err| {err:.3g} (SDPA bf16 {base:.3g})")
assert err <= 2 * base
flops = 4 * b_ * h_ * s_ * s_ * d_
print("warpdrive fa4_fwd        ", tflops(flops, bench_us(fa4_fwd, q, k, v)))
print("warpdrive fa4_fwd, pair  ", tflops(flops, bench_us(fa4_fwd, q, k, v, cluster=CtaPair())))
print("torch SDPA               ", tflops(flops, bench_us(sdpa)))

TMEM: 512 columns used, 512 allocated
  tmem S   [0, 128), [128, 256)
  tmem O   [256, 384), [384, 512)
  tmem P   [64, 128), [192, 256) (inside S)

SMEM plan: 229376 B in 3 buffer(s), plus alignment padding
  O  BFloat16  65536 B  2 stages (mode -1)  align 1024  -> epi
  Q  BFloat16  65536 B  2 stages (mode -1)  align 1024  -> q
  K  BFloat16  98304 B  3 stages (mode -1)  align 1024  -> kv
  V  BFloat16  98304 B  3 stages (mode -1)  overlaps K  -> kv

Alias diagnostic (bug finder: reports potential hazards; never proves safety)
  P ~ S (tmem)  no hazard found among 24 conflicting annotated access pair(s)
  V ~ K (smem)  no hazard found among 20 conflicting annotated access pair(s)



fa4_fwd max |err| 0.000893 (SDPA bf16 0.000893)


warpdrive fa4_fwd            348.5 us    1577 TFLOP/s


warpdrive fa4_fwd, pair      328.5 us    1673 TFLOP/s


torch SDPA                   297.2 us    1850 TFLOP/s


### A hazard the diagnostic finds

`overlap_o_q=True` places the output staging buffer O over Q (FA4's `overlap_sO_sQ`). With
one item per CTA that is safe. Under a persistent schedule the load warp may land the next
item's Q while the epilogue's TMA store still reads O from the same bytes, unless something
orders them. `OverlapOQ(ordered=True)` adds that edge as a one-stage `q_free` channel
(epilogue → load, payload-free); `ordered=False` drops it. Both compile; only the diagnostic
tells them apart. (`trace_protocol = True` also keeps these notebook-defined subclasses out
of the disk cache: the example's compile helper builds traced kernels uncached.)

In [15]:
class OverlapOQ(FlashAttentionForward):
    def __init__(self, ordered: bool):
        super().__init__(overlap_o_q=True)
        self.ordered = ordered
        self.q_free = Ring(1, Threads, Threads)  # epilogue: "O stored"; load: next Q may land
        self.work = WorkStream(mapping=Raster(along="m"), schedule=StaticPersistent())
        self.trace_protocol = True

    @role(warps=14, regs=48)
    @cute.jit
    def load(
        self, params: Fa4Params, *, q: Producer, kv: Producer, q_free: Consumer, work: WorkCursor
    ):
        rank = self.cluster.rank()
        thr = params.mma_qk.get_slice(rank), params.mma_pv.get_slice(rank)
        while work.valid:
            q, kv = self.load_item(params, q, kv, thr, work.item)
            if cutlass.const_expr(self.ordered):
                with q_free.consume():
                    pass
            work.advance()
        return q, kv, q_free, work

    @role(warps=13, regs=48)
    @cute.jit
    def epilogue(self, params: Fa4Params, *, epi: Consumer, q_free: Producer, work: WorkCursor):
        rank = self.cluster.rank()
        while work.valid:
            epi = self.epilogue_item(params, epi, rank, work.item)
            if cutlass.const_expr(self.ordered):
                with q_free.produce():
                    pass
            work.advance()
        return epi, q_free, work


for ordered in (False, True):
    op = OverlapOQ(ordered)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        compile_fa4_fwd(op)
    print(f"ordered={ordered}: {len(caught)} warning(s)", *(w.message for w in caught))
    text = op.describe()
    in_pair = False
    for line in text[text.index("Alias diagnostic") :].split("\n\n")[0].splitlines()[1:]:
        if not line.startswith("    "):  # a pair's first line; its findings are indented
            in_pair = "O ~ Q" in line
        if in_pair:
            print(line)
    print()

ordered=False: 1 warning(s) warpdrive: alias diagnostic flagged O~Q in OverlapOQ (no error raised; see describe()): unverified: correction write O[0] by threads @ fa4_fwd.py:581 before s fa4_fwd.py:579: no fence.proxy.async between a generic access and a later umma access (channel smem_async fence or fence_async_shared()) [proxy]
  O ~ Q (smem)  potential hazard
      unverified: correction write O[0] by threads @ fa4_fwd.py:581 before s fa4_fwd.py:579: no fence.proxy.async between a generic access and a later umma access (channel smem_async fence or fence_async_shared()) [proxy]
      potential hazard: correction write O[0] by threads @ fa4_fwd.py:581 (work item 0) and load write Q[0] by tma @ fa4_fwd.py:292 (work item 1): no channel path orders them [channel]
      unverified: correction write O[1] by threads @ fa4_fwd.py:581 before s fa4_fwd.py:579: no fence.proxy.async between a generic access and a later umma access (channel smem_async fence or fence_async_shared()) [proxy]
      

ordered=True: 0 warning(s)
  O ~ Q (smem)  no hazard found among 8 conflicting annotated access pair(s)



The finding names both accesses, their source lines and work items, and the missing edge
(`no channel path orders them`). Each `with q_free...: pass` is a pure synchronization
edge: a channel with no payload is just a pair of barriers.

## 6. The checkers

Tracing a role body checks the local protocol: every handle closes exactly once in the
region that opened it, producers commit and consumers release, every state argument comes
back, work loops have the drain shape, try tokens and previews belong to their epoch, and
recognized TMA copies match the declared bytes. Violations raise `ProtocolError` at compile
time, naming the role, channel and source lines (a notebook cell's lines appear under
IPython's hashed file name for the cell).

The TMA check compares the boxes of the copies that complete on `h.barrier` with the
channel's `Tma(bytes)`. Declaring half the GEMM's stage bytes would let the FULL barrier
complete when only half the stage has landed.

In [16]:
class BuggyRelay(Relay):
    def __init__(self, bug: str):
        super().__init__()
        self.bug = bug

    @role(warps=0)
    @cute.jit
    def producer(self, ctx, *, ch: Producer):
        lane = cute.arch.lane_idx()
        for e in cutlass.range(ctx.n, unroll=1):
            h = ch.acquire()
            h.data[lane] = e * WORDS + lane
            if cutlass.const_expr(self.bug != "never committed"):
                h.commit()
            if cutlass.const_expr(self.bug == "committed twice"):
                h.commit()
            if cutlass.const_expr(self.bug == "released by the producer"):
                h.release()
        if cutlass.const_expr(self.bug == "end not returned"):
            return None
        return ch


for bug in ("never committed", "committed twice", "released by the producer", "end not returned"):
    try:
        build_relay(BuggyRelay(bug))
    except ProtocolError as error:
        print(f"{bug}:\n  {error}\n")


class HalfBytesGemm(Gemm):
    def __init__(self):
        super().__init__()
        buffers = {"A": self.smem_plan["A"], "B": self.smem_plan["B"]}
        self.ab = Ring(3, Tma((128 + 256) * 64), Umma, buffer=buffers)  # forgot "* 2" for bf16


try:
    compile_tvm_ffi(HalfBytesGemm(), *gemm_fakes(), name="tour_half_bytes")
except ProtocolError as error:
    print(f"half the TMA bytes:\n  {error}")

never committed:
  producer: handles never closed: ch[i] acquire opened at 3915362516.py:11



committed twice:
  producer: ch[i] acquire closed twice: commit at 3915362516.py:14, then commit at 3915362516.py:16

released by the producer:
  producer: release() on a ch[i] acquire handle; producer handles commit or retire, consumer handles release

end not returned:
  producer: return all warpdrive state arguments; missing ch (for example, return ch)



half the TMA bytes:
  load: ab[i] acquire opened at gemm.py:111: the TMA copies at gemm.py:113, gemm.py:116 deliver 49152 bytes, but the layer arms 24576 (channel ab, Tma(24576)), so the stage's full barrier completes before every copy lands; declare Tma(49152) or fix the copies


`trace_protocol = True` (before the first `describe()` or compile) records every traced
acquire/commit/wait/release site. It is a list of code sites, not runtime events: a wait in
a loop appears once, indented by loop depth.

In [17]:
traced = Relay(style="token")
traced.trace_protocol = True
build_relay(traced)
show(traced, "Traced")

Traced protocol
  role producer:
      ch[i]     acquire  @ 2037731008.py:17
      ch[i]     commit   @ 2037731008.py:19
  role consumer:
      ch[i]     try_wait @ 2037731008.py:29
      ch[i]     wait     @ 2037731008.py:30
      ch[i]     release  @ 2037731008.py:32


What the checks do not do: compare the two sides' epoch counts in dynamic loops, prove the
dependency graph can make progress, or validate raw code. A producer that commits `n + 1`
times against a consumer that waits `n` times compiles; so does a `with` placed under a
lane-dependent `if` (predicate the data access, not the handoff). The
[safety contract](../GUARANTEES.md) lists what each check means.

## 7. Dropping down

Each escape hatch hands one piece of the protocol back to you; `describe()` records it.

| Hatch | You take over |
|---|---|
| `h.commit(signal=False)` / `h.release(signal=False)` | one signal and its fences; `h.barrier`, `h.empty_barrier`, `h.index`, `h.phase` give the raw state |
| `Channel(override={"empty_counts": 1, ...})` | a derived count, `tx` or mask |
| `Ring(..., external="consumer")` | a whole side, driven from raw code via `self.barrier_ptr(name)` |
| `Barriers(n, counts=...)` | a reserved mbarrier set with no checks (e.g. for a `cutlass.pipeline`) |
| `@role(..., raw=True)` | a role the layer only places, budgets and dispatches |
| `with unchecked(check, reason=...)` | one trace-time check at one site |

[`examples/raw_interop.py`](../examples/raw_interop.py) mixes them on one ring: a managed
producer, a raw consumer driving the external side with `cute.arch.mbarrier_wait`/`arrive`
and its own `PipelineState` (one arrival per stage, declared with `override`), and a raw
watcher that joins a `NamedBarrier` through its native ID.

In [18]:
from transformer_nuggets.cute.warpdrive.examples.raw_interop import (
    RawInterop,
    compile_raw_interop,
    raw_interop,
)

for n in (1, 2, 9):
    wrong, total = raw_interop(n).tolist()
    assert (wrong, total) == (0, sum(range(64 * n))), (n, wrong, total)
print("raw_interop matches for n = 1, 2, 9\n")
interop = RawInterop()
compile_raw_interop(interop)  # describe() reports the named-barrier users after a compile
show(interop, "Roles", "Channels", "Named")
print("\n" + inspect.getsource(RawInterop.consumer))

raw_interop matches for n = 1, 2, 9

Roles
  warps  role      regs  cta  raw
  0      producer     -  all
  1-2    consumer     -  all  raw
  3      watcher      -  all  raw

Channels
  ring  Ring(2)  producer -> raw code
        signal    Threads -> Threads
        full      32 x2
        empty     1 x2
        external  consumer side driven by raw code; empty counts are the override's (unverified)

Named barriers
  read  id 3  64 threads  sync by consumer
  done  id 4  96 threads  arrive by consumer; native id by watcher; unverified: roles calling it have 64 of 96 threads; native code reading .id (watcher) must supply the rest

    @role(warps=range(1, 3), raw=True)
    @cute.jit
    def consumer(self, ctx):
        t = cute.arch.thread_idx()[0] - CONSUMER_FIRST_THREAD
        bad, total = cutlass.Int32(0), cutlass.Int32(0)
        full = self.barrier_ptr("ring")  # full barriers at [0, n), empty at [n, 2n)
        empty = full + self.stages
        state = pipeline.make_pipeline_sta

`describe()` marks both raw boundaries: the external side's count is the override's
(`unverified`), and `done`'s participation is unverified because the watcher's arrivals come
from a native `bar.sync` the layer cannot see.

## 8. How the real examples map onto the nouns

### `persistent_gemm.py`

| Noun | Where |
|---|---|
| `WarpSpecialized` | `PersistentGemm`; static choices (schedule, mapping, cluster, stages, `tile_n`) are constructor arguments, so they key the compile |
| `@role` / `RoleConfig` | `load` (warp 4, 40 regs), `mma` (warp 5, `cta="leader"`), `epilogue` (warps 0–3, 160); `roles=` overrides |
| `Ring` + `Tma`/`Umma` | `ab = Ring(stages, Tma(stage_bytes, multicast=...), Umma, buffer={"A", "B"})` |
| `Ring` + `Umma`/`Threads` | `acc = Ring(2, Umma, Threads)`: two TMEM accumulators; `a = acc.wait()` ... `a.release()` explicitly, so `early_release` can hand TMEM back before the last TMA store |
| `SmemPlan` | A, B (bound to `ab`) and C (epilogue staging, used outside any channel) |
| `NamedBarrier` | `epi_sync` (128 threads, `aligned=True`) around each staged TMA store |
| `TmemPlan` / `Tmem` | `region("acc", tile_n, acc_stages)`; `mma` and `epilogue` take `tmem: Tmem` |
| `WorkStream` | `Raster(swizzle=8)` + any schedule; `mma_tile_mn` offsets a cluster item by the pair's position |
| `Cluster` | `self.cluster.multicast("n"/"m")` builds TMA partitions and masks; `layout_vmnk()` and `cta_group` build the atoms |
| Raw `cute.arch` | `fence_view_async_tmem_load`, `fence_view_async_shared`, `cp_async_bulk_commit_group`/`wait_group` in the epilogue |

### `fa4_fwd.py`

| Noun | Where |
|---|---|
| `Slots` + `Instance` | `q` (two Q tiles), `s`, `p_last`, `stats`, `epi`: one slot per Q tile; `softmax` is two instances (`warps=(range(4), range(4, 8))`), each bound to its slot (`softmax[0]@0`) |
| Fan-out / join | `s = Slots(2, Umma, Threads, consumer_roles=("softmax", "correction"), ...)`: one MMA commit wakes both consumers; EMPTY joins them (256 arrivals) |
| Overlapping handles | the MMA holds `v = kv.wait()` (V(i-1)) and `k = kv.wait()` (K(i)), releasing V before K: explicit `wait`/`release`, not nested `with` |
| Fences | `s` releases with `release_fence="tmem_store"` (P and rescaled O written by `tcgen05.st`); `epi` commits with `"smem_async"` for the TMA store |
| Payload | `stats = Slots(2, Threads, Threads, payload=(Float32, (128, 2)))` |
| Aliases | V `overlaps` K; TMEM P is an alias inside S; every access is annotated for the diagnostic |
| `Turnstile` | `mufu` around the two softmax instances' exp2 (disabled by default, as in FA4) |
| `CtaPair` | `cluster=CtaPair()`: 512-row items, leader-only MMA, pair-scoped q/kv/s; stats/epi stay CTA-local |

In [19]:
show(fa4, "FlashAttentionForward", "Roles", "Channels", "Turnstiles")

FlashAttentionForward: 16 warps, 512 threads
  cluster  single CTA, tcgen05 CtaGroup.ONE
  params   Fa4Params

Roles
  warps  role        regs  cta  raw
  0-3    softmax[0]   192  all
  4-7    softmax[1]   192  all
  8-11   correction    80  all
  12     mma           48  all
  13     epilogue      48  all
  14     load          48  all
  15     (padding)     48

Channels
  q       Slots(2) load -> mma
          signal    Tma(32768 B) -> Umma
          full      1 x2, tx 32768 B
          empty     1 x2
          storage   buffer Q
          tma       verified CtaGroup.ONE, 32768 B/stage per CTA @ fa4_fwd.py:293
  kv      Ring(3)  load -> mma
          signal    Tma(32768 B) -> Umma
          full      1 x3, tx 32768 B
          empty     1 x3
          storage   buffer K, V
          tma       verified CtaGroup.ONE, 32768 B/stage per CTA @ fa4_fwd.py:288,299,307,314
  s       Slots(2) mma -> correction, softmax[0]@0, softmax[1]@1
          signal    Umma -> Threads
          full     

## Verb cheat sheet

| Verb | Side | Effect |
|---|---|---|
| `h = end.acquire()` / `end.acquire(j)` | producer | wait for an empty stage (slot `j`); arms TMA bytes; advances the end |
| `h.commit()` / `h.retire()` | producer | publish / close a final unused acquire |
| `with end.produce() as h:` | producer | acquire + commit |
| `h = end.wait()` / `end.wait(j)` | consumer | wait for a full stage; advances the end |
| `h.release()` | consumer | hand the stage back (after the side's fences) |
| `with end.consume() as h:` | consumer | wait + release |
| `end.try_wait()` / `try_acquire()` → `Token`; `wait(token=)` | either | probe once, block only if needed |
| `end.peek()` → `Preview` | consumer | read the next epoch without taking it |
| `end.skip(n)` | exclusive consumer | advance past epochs another role owns |
| `end.views(A=StageView(...))`, `h.view("A", ...)`, `h.data` | either | stage-projected storage |
| `h.barrier`, `h.index`, `h.phase`, `signal=False` | either | raw state for `cute.arch` code |
| `work.valid`, `work.item`, `work.advance()` | any role | drain the work stream |
| `nb.sync()`, `nb.arrive()`, `with seat.turn():` | any role | named-barrier rendezvous, turnstile |
| `op.describe()`, `op.trace_protocol`, `op.alias_check`, `op.enable_iket()` | host | inspect, trace, check aliases, instrument |

Next: [Write a kernel](writing-kernels.md) for the authoring path,
[GUARANTEES.md](../GUARANTEES.md) for what the checks mean, and the topic guides:
[roles](roles.md), [channels](channels.md), [storage](storage.md),
[scheduling](scheduling.md), [clusters](clusters.md), [dropping down](dropping_down.md),
[tracing](tracing.md).